# DermaViT Colab Runner - Updated for Metadata Fusion
This notebook mounts Google Drive, syncs code from GitHub, prepares dataset once, and runs full training.

**Updates:**
- Metadata fusion support (age, sex, localization)
- Class-adaptive λ in explainability
- Hyperparameter ablation study mode
- Top-3 accuracy and Dice-based XAI evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

RESEARCH_DIR = "/content/drive/MyDrive/DermaViT_Research"
REPO_URL = "https://github.com/Nishi-Kanta-Paul/DermaViT-Dual-Branch-Fusion-for-Skin-Lesion-Classification.git"
WORKSPACE_DIR = "/content/DermaViT_Workspace"
DATA_ARCHIVE_CANDIDATES = [
    f"{RESEARCH_DIR}/data.zip",
    f"{RESEARCH_DIR}/HAM10000.zip",
    f"{RESEARCH_DIR}/dataset.zip",
]
DATA_ROOT = f"{RESEARCH_DIR}/data"
OUTPUT_ROOT = f"{RESEARCH_DIR}/outputs"

print("RESEARCH_DIR =", RESEARCH_DIR)
print("DATA_ROOT (expected) =", DATA_ROOT)
print("DATA_ARCHIVE_CANDIDATES =", DATA_ARCHIVE_CANDIDATES)
print("OUTPUT_ROOT =", OUTPUT_ROOT)

RESEARCH_DIR = /content/drive/MyDrive/DermaViT_Research
DATA_ZIP = /content/drive/MyDrive/DermaViT_Research/HAM10000.zip
DATA_ROOT = /content/drive/MyDrive/DermaViT_Research/HAM10000
OUTPUT_ROOT = /content/drive/MyDrive/DermaViT_Research/outputs


In [ ]:
%cd /content
!rm -rf DermaViT_Workspace
!git clone "$REPO_URL" DermaViT_Workspace
%cd /content/DermaViT_Workspace
!git rev-parse --short HEAD

/content
Cloning into 'DermaViT_Workspace'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 62 (delta 31), reused 51 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 28.91 KiB | 9.64 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/DermaViT_Workspace
e2532b7


In [ ]:
# Dataset discovery + optional unzip (supports both new and legacy layouts)
import glob
import os
import shutil
import zipfile

metadata_matches = glob.glob(f"{RESEARCH_DIR}/**/metadata.csv", recursive=True)
legacy_matches = glob.glob(f"{RESEARCH_DIR}/**/GroundTruth.csv", recursive=True)

if not metadata_matches and not legacy_matches:
    archive_path = next((p for p in DATA_ARCHIVE_CANDIDATES if os.path.isfile(p)), None)
    assert archive_path is not None, (
        "No dataset found. Checked metadata.csv / GroundTruth.csv and archives: "
        f"{DATA_ARCHIVE_CANDIDATES}"
    )
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(RESEARCH_DIR)

    metadata_matches = glob.glob(f"{RESEARCH_DIR}/**/metadata.csv", recursive=True)
    legacy_matches = glob.glob(f"{RESEARCH_DIR}/**/GroundTruth.csv", recursive=True)

assert metadata_matches or legacy_matches, "Dataset label CSV not found after extraction."

if metadata_matches:
    DATA_ROOT = os.path.dirname(metadata_matches[0])
else:
    DATA_ROOT = os.path.dirname(legacy_matches[0])
    legacy_csv = os.path.join(DATA_ROOT, "GroundTruth.csv")
    metadata_csv = os.path.join(DATA_ROOT, "metadata.csv")
    if not os.path.isfile(metadata_csv):
        shutil.copy2(legacy_csv, metadata_csv)
        print(f"Created metadata.csv from legacy GroundTruth.csv at: {DATA_ROOT}")

assert os.path.isdir(os.path.join(DATA_ROOT, "images")), f"images folder missing in: {DATA_ROOT}"
assert os.path.isfile(os.path.join(DATA_ROOT, "metadata.csv")), f"metadata.csv missing in: {DATA_ROOT}"

os.environ["DERMAVIT_DATA_ROOT"] = DATA_ROOT
os.environ["DERMAVIT_OUTPUT_DIR"] = OUTPUT_ROOT

print("DERMAVIT_DATA_ROOT =", os.environ["DERMAVIT_DATA_ROOT"])
print("DERMAVIT_OUTPUT_DIR =", os.environ["DERMAVIT_OUTPUT_DIR"])

DERMAVIT_DATA_ROOT = /content/drive/MyDrive/DermaViT_Research
DERMAVIT_OUTPUT_DIR = /content/drive/MyDrive/DermaViT_Research/outputs


In [ ]:
%cd /content/DermaViT_Workspace
!if [ -f "requirements.txt" ]; then \
    python -m pip install -r requirements.txt; \
  elif [ -f "DermaViT/requirements.txt" ]; then \
    python -m pip install -r DermaViT/requirements.txt; \
  else \
    echo "No requirements file found"; \
    find . -maxdepth 4 -type f -name "requirements*.txt"; \
    exit 1; \
  fi

/content/DermaViT_Workspace


In [ ]:
import os
os.environ["DERMAVIT_DATA_ROOT"] = DATA_ROOT
os.environ["DERMAVIT_OUTPUT_DIR"] = OUTPUT_ROOT
print('DERMAVIT_DATA_ROOT =', os.environ['DERMAVIT_DATA_ROOT'])
print('DERMAVIT_OUTPUT_DIR =', os.environ['DERMAVIT_OUTPUT_DIR'])

DERMAVIT_DATA_ROOT = /content/drive/MyDrive/DermaViT_Research
DERMAVIT_OUTPUT_DIR = /content/drive/MyDrive/DermaViT_Research/outputs


In [ ]:
# Run DermaViT pipeline
# Set ABLATION_MODE=True in src/config.py to run hyperparameter ablation study
%cd /content/DermaViT_Workspace

# Option 1: Run main DermaViT pipeline
!python src/main.py

# Option 2: Run with ablation study (uncomment to use)
# Edit src/config.py: ABLATION_MODE = True
# !python src/main.py

# Option 3: Run complete pipeline with baselines
# !chmod +x scripts/train.sh
# !./scripts/train.sh

Streaming output truncated to the last 5000 lines.
  Train:  36% 91/251 [00:48<01:08,  2.33it/s, acc=92.4%, loss=0.1046]DEBUG: f_local dim=2 shape=torch.Size([32, 1280])
DEBUG: f_global dim=2 shape=torch.Size([32, 768])
  Train:  37% 92/251 [00:48<01:06,  2.39it/s, acc=92.4%, loss=0.1575]DEBUG: f_local dim=2 shape=torch.Size([32, 1280])
DEBUG: f_global dim=2 shape=torch.Size([32, 768])
  Train:  37% 93/251 [00:48<01:09,  2.26it/s, acc=92.5%, loss=0.0312]DEBUG: f_local dim=2 shape=torch.Size([32, 1280])
DEBUG: f_global dim=2 shape=torch.Size([32, 768])
  Train:  37% 94/251 [00:49<01:13,  2.12it/s, acc=92.5%, loss=0.0219]DEBUG: f_local dim=2 shape=torch.Size([32, 1280])
DEBUG: f_global dim=2 shape=torch.Size([32, 768])
  Train:  38% 95/251 [00:50<01:25,  1.82it/s, acc=92.5%, loss=0.0531]DEBUG: f_local dim=2 shape=torch.Size([32, 1280])
DEBUG: f_global dim=2 shape=torch.Size([32, 768])
  Train:  38% 96/251 [00:51<01:36,  1.61it/s, acc=92.6%, loss=0.0681]DEBUG: f_local dim=2 shape=torch.Si

In [ ]:
# Optional: quick output check
!ls -lah "$OUTPUT_ROOT"
!find "$OUTPUT_ROOT" -maxdepth 3 -type f | head -n 30

## Later Runs (No Re-upload)
For later code updates, run only: mount drive -> clone/pull -> install (if needed) -> set env vars -> run_all.
Dataset discovery auto-skips extraction if `metadata.csv` already exists and also supports legacy `GroundTruth.csv`.